# Applying SMOTE to anime genre classification


In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score

In [33]:
# SETTINGS
target_col = 'genres'
MIN_CLASS_SIZE = 30
RANDOM_STATE = 42
TEST_SIZE = 0.2
K_NEIGHBORS = 5 

In [34]:
anime = pd.read_csv('cleaned_animes.csv')
anime = anime.dropna(subset=['synopsis', target_col])

anime['title_english'] = anime['title_english'].fillna('')
anime['text'] = anime['title_english'] + ' ' + anime['synopsis']

In [35]:
# KEEP ONLY CLASSES WITH ENOUGH SAMPLES TO UPSAMPLE
class_counts = anime[target_col].value_counts()
keep_classes = class_counts[class_counts >= MIN_CLASS_SIZE].index

df = anime[anime[target_col].isin(keep_classes)].reset_index(drop=True)

In [36]:
# TRAIN / TEST SPLIT
X_text = df['text']
y = df[target_col]

X_train_text, X_test_text, y_train, y_test = train_test_split(X_text, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

In [37]:
# TF-IDF VECTORIZATION
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))

X_train = vectorizer.fit_transform(X_train_text) 
X_test = vectorizer.transform(X_test_text) 

X_train.shape, X_test.shape

((6152, 5000), (1538, 5000))

In [38]:
# UPSAMPLE ALL CLASSES WITH SMOTE
smote = SMOTE(sampling_strategy='not majority', k_neighbors=K_NEIGHBORS, random_state=RANDOM_STATE)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

dist = pd.DataFrame({
    'original_train': y_train.value_counts(),
    'smote_train': pd.Series(y_train_resampled).value_counts(),
}).sort_values('original_train', ascending=False)

/Users/braydenhumpherys/anaconda3/lib/python3.11/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


In [39]:
# TRAINING MODELS ON NORMAL VS SMOTE TRAINING DATA
model_normal = LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
model_normal.fit(X_train, y_train)

model_resampled = LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
model_resampled.fit(X_train_resampled, y_train_resampled)

# predict on the same untouched test set
y_pred_normal = model_normal.predict(X_test)
y_pred_resampled = model_resampled.predict(X_test)

In [40]:
# PERFORMANCE COMPARISON
results = pd.DataFrame({
    'Normal data': [
        accuracy_score(y_test, y_pred_normal),
        f1_score(y_test, y_pred_normal, average='weighted'),
    ],
    'SMOTE data': [
        accuracy_score(y_test, y_pred_resampled),
        f1_score(y_test, y_pred_resampled, average='weighted'),
    ],
}, index=['Accuracy', 'Weighted F1'])
results['Difference (SMOTE - Normal)'] = results['SMOTE data'] - results['Normal data']
results.round(4)

,Normal data,SMOTE data,Difference (SMOTE - Normal)
Accuracy,0.3440,0.3784,0.0345
Weighted F1,0.2761,0.3702,0.0941
